## Preprocess Dataset Notebook 

In [105]:
#Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import mne
from scipy.stats import mannwhitneyu
import os

os.chdir("/Users/diego/Desktop/Master Neuro/M2/Internship_Valencia/multimodal-exist/")

In [106]:
#Load Data
df = pd.read_parquet("./data/processed/train_base.parquet")
df.head()

,id,lang,text,image_file,split,ET_raw,HR_raw,EEG_raw,task21_hard,task21_valid_hard,task21_soft,task22_hard,task22_valid_hard,task22_soft,task23_hard,task23_valid_hard,task23_soft,task23_num_hard_labels
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,1.000000,1.0,True,0.833333,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,{'IDEOLOGICAL-INEQUALITY': 0.16666666666666666...,2
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",NaN,False,0.500000,1.0,True,1.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,{'IDEOLOGICAL-INEQUALITY': 0.16666666666666666...,2
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,0.833333,1.0,True,1.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,"{'IDEOLOGICAL-INEQUALITY': 0.0, 'MISOGYNY-NON-...",1
3,110593,es,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",0.0,True,0.166667,NaN,False,0.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",False,"{'IDEOLOGICAL-INEQUALITY': 0.0, 'MISOGYNY-NON-...",0
4,110946,es,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,0.666667,1.0,True,1.000000,"{'IDEOLOGICAL-INEQUALITY': 0, 'MISOGYNY-NON-SE...",True,"{'IDEOLOGICAL-INEQUALITY': 0.0, 'MISOGYNY-NON-...",1


In [107]:
df["ET_raw"].iloc[0]

{'EN1': None,
 'EN2': None,
 'EN3': None,
 'EN4': None,
 'EN5': None,
 'EN6': None,
 'EN7': None,
 'ES1': {'3d_eye_states_pupil diameter left [mm]_max': 3.0798,
  '3d_eye_states_pupil diameter left [mm]_mean': 2.761,
  '3d_eye_states_pupil diameter left [mm]_min': 1.7061,
  '3d_eye_states_pupil diameter left [mm]_std': 0.1419,
  '3d_eye_states_pupil diameter right [mm]_max': 3.1986,
  '3d_eye_states_pupil diameter right [mm]_mean': 2.6514,
  '3d_eye_states_pupil diameter right [mm]_min': 1.8788,
  '3d_eye_states_pupil diameter right [mm]_std': 0.2088,
  'blinks_count': 2.361,
  'blinks_duration_max_ns': 280895959.54,
  'blinks_duration_mean_ns': 249338268.5011,
  'blinks_duration_min_ns': 222131939.4638,
  'blinks_duration_std_ns': 25269826.3622,
  'fixations_count': 28.8564,
  'fixations_duration_max_ns': 1030753083.594,
  'fixations_duration_mean_ns': 323316277.679,
  'fixations_duration_min_ns': 101753233.905,
  'fixations_duration_std_ns': 232424711.8143,
  'reaction_time': 10080.0

## Eye Tracking Agregation

In [108]:
ET_FEATURE_MAP = {
    "reaction_time": "et_reaction_time",                 # We transform ms -> s
    "fixations_count": "et_fixations",                   # We include the count
    "fixations_duration_mean_ns": "et_fixation_duration",# We transform ns -> ms
    "saccades_count": "et_saccades"                      # We include the count
    }

In [109]:
def aggregate_et_features(et_raw):
    
    # output template
    result = {"et_n_users": 0}
    
    # initialize all features with NaN
    for feat_name in ET_FEATURE_MAP.values():
        result[f"{feat_name}_mean"] = np.nan
        result[f"{feat_name}_std"] = np.nan
    
    if not isinstance(et_raw, dict):
        return result
    
    # keep only valid users with dict features
    valid_users = {user: feats for user, feats in et_raw.items()if isinstance(feats, dict)}
    result["et_n_users"] = len(valid_users)
    
    if len(valid_users) == 0:
        return result
    
    # collect values per feature across users
    collected = {k: [] for k in ET_FEATURE_MAP.keys()}
    
    for _, feats in valid_users.items():
        for raw_key in ET_FEATURE_MAP.keys():
            value = feats.get(raw_key, None)
            if value is not None:
                collected[raw_key].append(value)
    
    # miliseconds to seconds 
    if len(collected["reaction_time"]) > 0:
        collected["reaction_time"] = [x / 1000 for x in collected["reaction_time"]]
    
    # nanoseconds to milliseconds
    if len(collected["fixations_duration_mean_ns"]) > 0:
        collected["fixations_duration_mean_ns"] = [x / 1e6 for x in collected["fixations_duration_mean_ns"]]  
    
    # aggregate
    for raw_key, feat_name in ET_FEATURE_MAP.items():
        values = pd.to_numeric(pd.Series(collected[raw_key]), errors="coerce").dropna()
        
        if len(values) > 0:
            result[f"{feat_name}_mean"] = values.mean()
            result[f"{feat_name}_std"] = values.std()
    
    return result

In [110]:
et_agg_df = df["ET_raw"].apply(aggregate_et_features).apply(pd.Series)
et_agg_df.head(3)

,et_n_users,et_reaction_time_mean,et_reaction_time_std,et_fixations_mean,et_fixations_std,et_fixation_duration_mean,et_fixation_duration_std,et_saccades_mean,et_saccades_std
0,2.0,13.394003,4.686700,42.92820,19.900530,290.553894,46.333007,42.65420,20.288025
1,3.0,14.951000,8.539386,48.00000,25.238859,272.211591,32.506004,47.00000,25.238859
2,2.0,11.928333,3.162653,40.33335,3.771213,259.975251,129.980125,39.66665,3.299855


In [111]:
df = pd.concat([df, et_agg_df], axis=1)
et_cols = [c for c in df.columns if c.startswith("et_")]
df.head(3)

,id,lang,text,image_file,split,ET_raw,HR_raw,EEG_raw,task21_hard,task21_valid_hard,...,task23_num_hard_labels,et_n_users,et_reaction_time_mean,et_reaction_time_std,et_fixations_mean,et_fixations_std,et_fixation_duration_mean,et_fixation_duration_std,et_saccades_mean,et_saccades_std
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,2,2.0,13.394003,4.686700,42.92820,19.900530,290.553894,46.333007,42.65420,20.288025
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",NaN,False,...,2,3.0,14.951000,8.539386,48.00000,25.238859,272.211591,32.506004,47.00000,25.238859
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,1,2.0,11.928333,3.162653,40.33335,3.771213,259.975251,129.980125,39.66665,3.299855


## EEG Agregation

In [112]:
EEG_BANDS = ["Alpha", "Beta", "Theta", "Gamma"]

#Change if necessary based on the actual channel names in the dataset
EEG_REGION_MAP = {
    "frontal": [0, 1, 2, 3],
    "central": [4, 5, 6, 7],
    "parietal": [8, 9, 10, 11],
    "occipital": [12, 13, 14, 15]}

# These are the features we will use for modeling, based on the previous analysis (01_Sensory_Analysis)
EEG_FINAL_FEATURES = [
    "eeg_alpha_global",
    "eeg_beta_global",
    "eeg_theta_global",
    "eeg_gamma_global",
    "eeg_beta_minus_alpha_global",
    "eeg_theta_minus_alpha_global",
    "eeg_alpha_frontal",
    "eeg_alpha_occipital",
    "eeg_alpha_central",
    "eeg_alpha_parietal"]


In [113]:
def extract_eeg_user_features(user_feats):
    """
    Compute compact EEG features for one valid user.
    Assumes all required EXG_Channel_*_{Band}_power keys exist.
    """
    result = {}
    
    # Global band means across all 16 channels
    for band in EEG_BANDS:
        cols = [f"EXG_Channel_{ch}_{band}_power" for ch in range(16)]
        vals = pd.to_numeric(pd.Series([user_feats.get(c, np.nan) for c in cols]), errors="coerce").dropna()
        result[f"eeg_{band.lower()}_global"] = vals.mean() if len(vals) > 0 else np.nan
    
    # Regional alpha means
    for region_name, ch_list in EEG_REGION_MAP.items():
        cols = [f"EXG_Channel_{ch}_Alpha_power" for ch in ch_list]
        vals = pd.to_numeric(pd.Series([user_feats.get(c, np.nan) for c in cols]), errors="coerce").dropna()
        result[f"eeg_alpha_{region_name}"] = vals.mean() if len(vals) > 0 else np.nan
    
    # Stable contrasts
    result["eeg_beta_minus_alpha_global"] = (result["eeg_beta_global"] - result["eeg_alpha_global"]
        if pd.notna(result["eeg_beta_global"]) and pd.notna(result["eeg_alpha_global"])
        else np.nan)
    
    result["eeg_theta_minus_alpha_global"] = (result["eeg_theta_global"] - result["eeg_alpha_global"]
        if pd.notna(result["eeg_theta_global"]) and pd.notna(result["eeg_alpha_global"])
        else np.nan)
    
    return result

In [114]:
#Same function as the ET 
def aggregate_eeg_features(eeg_raw):
    result = {"eeg_n_users": 0}
    
    for feat in EEG_FINAL_FEATURES:
        result[f"{feat}_mean"] = np.nan
        result[f"{feat}_std"] = np.nan
    
    if not isinstance(eeg_raw, dict):
        return result
    
    valid_users = {
        user: feats for user, feats in eeg_raw.items()
        if isinstance(feats, dict)}
    
    result["eeg_n_users"] = len(valid_users)
    
    if len(valid_users) == 0:
        return result
    
    # Per-user compact features
    user_feature_rows = []
    for _, feats in valid_users.items():
        user_feature_rows.append(extract_eeg_user_features(feats))
    
    user_df = pd.DataFrame(user_feature_rows)
    
    for feat in EEG_FINAL_FEATURES:
        vals = pd.to_numeric(user_df[feat], errors="coerce").dropna()
        if len(vals) > 0:
            result[f"{feat}_mean"] = vals.mean()
            result[f"{feat}_std"] = vals.std()
    
    return result

In [115]:
eeg_agg_df = df["EEG_raw"].apply(aggregate_eeg_features).apply(pd.Series)
eeg_agg_df.head(3)

,eeg_n_users,eeg_alpha_global_mean,eeg_alpha_global_std,eeg_beta_global_mean,eeg_beta_global_std,eeg_theta_global_mean,eeg_theta_global_std,eeg_gamma_global_mean,eeg_gamma_global_std,eeg_beta_minus_alpha_global_mean,...,eeg_theta_minus_alpha_global_mean,eeg_theta_minus_alpha_global_std,eeg_alpha_frontal_mean,eeg_alpha_frontal_std,eeg_alpha_occipital_mean,eeg_alpha_occipital_std,eeg_alpha_central_mean,eeg_alpha_central_std,eeg_alpha_parietal_mean,eeg_alpha_parietal_std
0,2.0,-0.007325,0.393717,0.435972,0.851555,-0.268491,0.484921,0.047144,0.697066,0.443297,...,-0.261166,0.091204,-0.040800,0.034471,-0.641063,1.315166,0.177175,0.144886,0.475388,0.080345
1,2.0,0.123981,0.369013,0.287212,0.210461,0.035294,0.419102,0.067362,0.086674,0.163231,...,-0.088688,0.050090,0.760163,0.080310,-0.615987,0.946940,0.271213,0.597028,0.080538,1.045829
2,2.0,0.419137,0.328460,0.474569,0.914669,0.546509,0.525341,0.038828,0.271984,0.055431,...,0.127372,0.196881,0.198213,0.074759,0.285050,0.172110,0.324163,0.624039,0.869125,0.592449


In [116]:
df = pd.concat([df, eeg_agg_df], axis=1)
eeg_cols = [c for c in df.columns if c.startswith("eeg_")]
df.head(3)

,id,lang,text,image_file,split,ET_raw,HR_raw,EEG_raw,task21_hard,task21_valid_hard,...,eeg_theta_minus_alpha_global_mean,eeg_theta_minus_alpha_global_std,eeg_alpha_frontal_mean,eeg_alpha_frontal_std,eeg_alpha_occipital_mean,eeg_alpha_occipital_std,eeg_alpha_central_mean,eeg_alpha_central_std,eeg_alpha_parietal_mean,eeg_alpha_parietal_std
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,-0.261166,0.091204,-0.040800,0.034471,-0.641063,1.315166,0.177175,0.144886,0.475388,0.080345
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",NaN,False,...,-0.088688,0.050090,0.760163,0.080310,-0.615987,0.946940,0.271213,0.597028,0.080538,1.045829
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,0.127372,0.196881,0.198213,0.074759,0.285050,0.172110,0.324163,0.624039,0.869125,0.592449


## Heart Rate Aggregation

In [117]:
HR_FEATURE_MAP = {
    "garmin_hr_mean": "hr_mean",
    "garmin_hr_max": "hr_max",
    "garmin_hr_min": "hr_min"}

In [118]:
def aggregate_hr_features(hr_raw):
    
    #We initialize the output with 0 users and NaN for all features
    result = {"hr_n_users": 0}
    
    for feat_name in HR_FEATURE_MAP.values():
        result[f"{feat_name}_mean"] = np.nan
        result[f"{feat_name}_std"] = np.nan
    
    if not isinstance(hr_raw, dict):
        return result
    
    valid_users = {user: feats for user, feats in hr_raw.items() if isinstance(feats, dict)}
    result["hr_n_users"] = len(valid_users)
    
    if len(valid_users) == 0:
        return result
    
    collected = {k: [] for k in HR_FEATURE_MAP.keys()}
    
    for _, feats in valid_users.items():
        for raw_key in HR_FEATURE_MAP.keys():
            value = feats.get(raw_key, None)
            if value is not None:
                collected[raw_key].append(value)
    
    for raw_key, feat_name in HR_FEATURE_MAP.items():
        values = pd.to_numeric(pd.Series(collected[raw_key]), errors="coerce").dropna()
        
        if len(values) > 0:
            result[f"{feat_name}_mean"] = values.mean()
            result[f"{feat_name}_std"] = values.std()
    
    return result

In [119]:
hr_agg_df = df["HR_raw"].apply(aggregate_hr_features).apply(pd.Series)

print(hr_agg_df.shape)
hr_agg_df.head(3)

(3984, 7)


,hr_n_users,hr_mean_mean,hr_mean_std,hr_max_mean,hr_max_std,hr_min_mean,hr_min_std
0,2.0,62.63795,4.063389,64.626750,5.128999,60.839050,2.600809
1,3.0,59.74850,3.773102,61.666667,5.033223,57.333333,1.527525
2,2.0,64.20835,5.668663,65.000000,5.656854,63.500000,4.949747


In [120]:
df = pd.concat([df, hr_agg_df], axis=1)

hr_cols = [c for c in df.columns if c.startswith("hr_")]
print(hr_cols)
df.head(3)

['hr_n_users', 'hr_mean_mean', 'hr_mean_std', 'hr_max_mean', 'hr_max_std', 'hr_min_mean', 'hr_min_std']


,id,lang,text,image_file,split,ET_raw,HR_raw,EEG_raw,task21_hard,task21_valid_hard,...,eeg_alpha_central_std,eeg_alpha_parietal_mean,eeg_alpha_parietal_std,hr_n_users,hr_mean_mean,hr_mean_std,hr_max_mean,hr_max_std,hr_min_mean,hr_min_std
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,0.144886,0.475388,0.080345,2.0,62.63795,4.063389,64.626750,5.128999,60.839050,2.600809
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",NaN,False,...,0.597028,0.080538,1.045829,3.0,59.74850,3.773102,61.666667,5.033223,57.333333,1.527525
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,0.624039,0.869125,0.592449,2.0,64.20835,5.668663,65.000000,5.656854,63.500000,4.949747


### Final Order and Format

In [121]:
df = df.rename(columns={
    "task21_valid_hard": "task21_valid",
    "task22_valid_hard": "task22_valid",
    "task23_valid_hard": "task23_valid"})

In [122]:
#Task 2.3 Expanded and Renamed

names_23 = {
    "IDEOLOGICAL-INEQUALITY": "ideological_inequality",
    "STEREOTYPING-DOMINANCE": "stereotyping_dominance",
    "OBJECTIFICATION": "objectification",
    "SEXUAL-VIOLENCE": "sexual_violence",
    "MISOGYNY-NON-SEXUAL-VIOLENCE": "misogyny_non_sexual_violence"}

#Hard
task23_hard_df = df["task23_hard"].apply(pd.Series).rename(columns=names_23)
task23_hard_df = task23_hard_df.add_prefix("task23_hard_")
task23_hard_df.head(2)

#Soft
task23_soft_df = df_proc["task23_soft"].apply(pd.Series).rename(columns=names_23)
task23_soft_df = task23_soft_df.add_prefix("task23_soft_")
task23_soft_df.head(2)

#Combine and Clean
df = pd.concat([df, task23_hard_df, task23_soft_df], axis=1)
df = df.drop(columns=["task23_hard", "task23_soft"])
df.head()

,id,lang,text,image_file,split,ET_raw,HR_raw,EEG_raw,task21_hard,task21_valid,...,task23_hard_ideological_inequality,task23_hard_misogyny_non_sexual_violence,task23_hard_objectification,task23_hard_sexual_violence,task23_hard_stereotyping_dominance,task23_soft_ideological_inequality,task23_soft_misogyny_non_sexual_violence,task23_soft_objectification,task23_soft_sexual_violence,task23_soft_stereotyping_dominance
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,0,1,1,0,0,0.166667,0.500000,0.666667,0.000000,0.166667
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",NaN,False,...,0,0,0,1,1,0.166667,0.166667,0.166667,0.333333,0.500000
2,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,0,0,1,0,0,0.000000,0.166667,0.666667,0.166667,0.166667
3,110593,es,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",0.0,True,...,0,0,0,0,0,0.000000,0.000000,0.166667,0.000000,0.166667
4,110946,es,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,Train,"{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN4':...","{'EN1': None, 'EN2': None, 'EN3': None, 'EN5':...",1.0,True,...,0,0,1,0,0,0.000000,0.000000,0.500000,0.166667,0.000000


In [123]:
meta_cols = ["id", "text", "split"]

et_cols = sorted([c for c in df.columns if c.startswith("et_")])
eeg_cols = sorted([c for c in df.columns if c.startswith("eeg_")])
hr_cols = sorted([c for c in df.columns if c.startswith("hr_")])

raw_cols = ["ET_raw", "HR_raw", "EEG_raw"]

task21_cols = ["task21_valid", "task21_hard", "task21_soft"]
task22_cols = ["task22_valid", "task22_hard", "task22_soft"]

task23_hard_cols = sorted([c for c in df.columns if c.startswith("task23_hard_")])
task23_soft_cols = sorted([c for c in df.columns if c.startswith("task23_soft_")])
task23_cols = ["task23_valid"] + task23_hard_cols + task23_soft_cols


final_cols = (meta_cols +
              et_cols +
              eeg_cols +
              hr_cols +
              task21_cols +
              task22_cols +
              task23_cols)

df_final = df[final_cols].copy()

print(df_final.shape)
df_final.head(2)

(3984, 57)


,id,text,split,et_fixation_duration_mean,et_fixation_duration_std,et_fixations_mean,et_fixations_std,et_n_users,et_reaction_time_mean,et_reaction_time_std,...,task23_hard_ideological_inequality,task23_hard_misogyny_non_sexual_violence,task23_hard_objectification,task23_hard_sexual_violence,task23_hard_stereotyping_dominance,task23_soft_ideological_inequality,task23_soft_misogyny_non_sexual_violence,task23_soft_objectification,task23_soft_sexual_violence,task23_soft_stereotyping_dominance
0,110887,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,Train,290.553894,46.333007,42.9282,19.900530,2.0,13.394003,4.686700,...,0,1,1,0,0,0.166667,0.500000,0.666667,0.000000,0.166667
1,110466,Se necesita cuidadora para adulto mayor.... fo...,Train,272.211591,32.506004,48.0000,25.238859,3.0,14.951000,8.539386,...,0,0,0,1,1,0.166667,0.166667,0.166667,0.333333,0.500000


In [124]:
#Export
OUTPUT_DIR = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_final.to_parquet(os.path.join(OUTPUT_DIR, "train_model_ready.parquet"), index=False)
print(f"Training dataset saved to {os.path.join(OUTPUT_DIR, 'train_model_ready.parquet')}")

Training dataset saved to data/processed\train_model_ready.parquet
